# MedSAM2: segment a 3D CT by prompting a single slice

SAM 2 tracks objects across video frames using memory attention. A CT volume has the
same structure — consecutive slices barely differ — so one box on one slice can be
propagated through the whole stack.

**Runtime → Change runtime type → T4 GPU** before running anything.

- MedSAM2: https://github.com/bowang-lab/MedSAM2 · weights research/education only
- Paper: https://arxiv.org/abs/2504.03600

## 1 · Setup

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
%%capture
!pip install -q git+https://github.com/rekalantar/medsam2-3d-ct.git
!pip install -q SimpleITK imageio huggingface_hub
!git clone -q https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
%cd /content/MedSAM2
!pip install -q -e ".[dev]"
!bash download.sh

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt

from medsam2_ct import (build_predictor, dice, init_state, largest_component,
                        load_demo_case, per_slice_dice, plot_slices,
                        rotate_clockwise, save_gif, segment_volume)

CKPT = "/content/MedSAM2/checkpoints/MedSAM2_latest.pt"
assert os.path.exists(CKPT), "download.sh did not produce MedSAM2_latest.pt"

ROTATE = 1      # clockwise quarter-turns, display only — these volumes read sideways

predictor = build_predictor(config="configs/sam2.1_hiera_t512.yaml", checkpoint=CKPT)
print(f"checkpoint {os.path.getsize(CKPT) / 1e6:.0f} MB — ready")

## 2 · Load a case

The demo dataset ships per-case volumes with ground-truth masks, a few MB each.
`load_demo_case` fetches one, windows it to the abdominal soft-tissue range, and
places a box on the largest lesion cross-section.

The prompt comes from the label. That is how promptable models are normally
evaluated — it simulates a perfect user prompt, so what you measure is propagation
quality rather than whether a human drew a good box.

In [ ]:
case = load_demo_case("000009_03_01_036-048")

print(f"volume  {case['volume'].shape}  spacing {tuple(round(s, 2) for s in case['spacing'])}")
print(f"HU      {case['hu'].min():.0f} to {case['hu'].max():.0f}")
print(f"lesion  {case['span']} slices, largest at {case['key']}")
print(f"box     {case['box']}")

CT spans thousands of Hounsfield units; the encoder takes 8-bit. Windowing keeps the
soft-tissue range and discards the rest, so the lesion gets all 256 levels instead of a
handful.

Check the HU range above: if it already spans only a few hundred, the volume arrived
pre-clipped and windowing is a no-op on it.

## 3 · The prompt

In [ ]:
key = case["key"]
shown = rotate_clockwise(case["volume"], ROTATE)[key]
shown_truth = rotate_clockwise(case["truth"], ROTATE)[key]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(shown, cmap="gray")
axes[0].set_title(f"slice {key}")
axes[1].imshow(shown, cmap="gray")
axes[1].contour(shown_truth, levels=[0.5], colors="#3FC1C9", linewidths=1.5)
axes[1].set_title("ground truth")
for ax in axes: ax.axis("off")
plt.tight_layout()

## 4 · Propagate

`segment_volume` sweeps forward from the prompted slice and then backward. That second
sweep looks redundant — it isn't. The prompted slice sits in the middle of the lesion,
so forward-only stops halfway.

In [ ]:
truth = case["truth"]
masks = largest_component(segment_volume(predictor, case["volume"], case["box"], key))

print(f"prediction    {masks.sum():>7,} voxels   Dice {dice(truth, masks):.3f}")
print(f"ground truth  {truth.sum():>7,} voxels")
print(f"key slice                        Dice {dice(truth[key], masks[key]):.3f}")

**What the second sweep is worth.** Same prompt, forward only:

In [ ]:
state, _, _ = init_state(predictor, case["volume"])
predictor.add_new_points_or_box(
    inference_state=state, frame_idx=key, obj_id=1,
    box=np.asarray(case["box"], dtype=np.float32))

forward = np.zeros_like(masks)
for idx, _ids, logits in predictor.propagate_in_video(state):
    forward[idx] = (logits[0] > 0).cpu().numpy().squeeze()

print(f"both ways     {masks.sum():>7,} voxels   Dice {dice(truth, masks):.3f}")
print(f"forward only  {forward.sum():>7,} voxels   Dice {dice(truth, forward):.3f}"
      f"   ({forward.sum() / masks.sum():.0%})")

## 5 · Look at it

Every slice the lesion touches, cropped and rotated to read the way an axial slice
should. Cyan is ground truth, coral is the prediction.

In [ ]:
fig = plot_slices(case["volume"], masks, truth, rotate=ROTATE, ncols=7,
                  title=f"{case['name']} — Dice {dice(truth, masks):.3f}")
fig.savefig("slices.png", dpi=150, bbox_inches="tight")

And as an animation, which is where propagation failures show up.

In [ ]:
z = (masks | truth).any(axis=(1, 2)).nonzero()[0]
lo, hi = max(0, z.min() - 2), min(len(case["volume"]), z.max() + 3)

save_gif(case["volume"][lo:hi], masks[lo:hi], "propagation.gif", fps=4, rotate=ROTATE)
print(f"{os.path.getsize('propagation.gif') / 1e6:.2f} MB, {hi - lo} frames")

In [ ]:
import IPython.display

IPython.display.Image("propagation.gif")

## 6 · Where the error lives

Volume Dice is one number over slices that are not equally hard — the prompted slice
was seen directly, the rest were inferred. Scoring each against its distance from the
prompt shows where the cost actually falls.

In [ ]:
offsets, scores = per_slice_dice(truth, masks, key_slice=key)

for off, d in zip(offsets, scores):
    print(f"  {off:+3d}  {d:.3f}  {'#' * int(d * 40)}")

plt.figure(figsize=(8, 3.5))
plt.plot(offsets, scores, "o-", color="#3FC1C9", linewidth=2)
plt.axvline(0, color="#FF6B5B", linestyle="--", label="prompted slice")
plt.xlabel("slices from prompt"); plt.ylabel("Dice")
plt.ylim(0, 1.05); plt.legend(); plt.tight_layout()
plt.savefig("decay.png", dpi=150, bbox_inches="tight")

## 7 · A longer lesion

The case above spans a handful of slices, so propagation never travelled far. This one
spans many more, which is the real test.

In [ ]:
case2 = load_demo_case("000002_02_01_044-083")
masks2 = largest_component(
    segment_volume(predictor, case2["volume"], case2["box"], case2["key"]))

for c, m in ((case, masks), (case2, masks2)):
    off, sc = per_slice_dice(c["truth"], m, key_slice=c["key"])
    print(f"{c['name']}")
    print(f"   {c['span']:>3} slices   volume Dice {dice(c['truth'], m):.3f}"
          f"   key slice {dice(c['truth'][c['key']], m[c['key']]):.3f}")
    print(f"   best slice at offset {off[sc.argmax()]:+d}")
    print()

In [ ]:
off2, sc2 = per_slice_dice(case2["truth"], masks2, key_slice=case2["key"])

plt.figure(figsize=(9, 3.5))
plt.plot(offsets, scores, "o-", color="#3FC1C9", linewidth=2,
         label=f"{case['span']} slices")
plt.plot(off2, sc2, "o-", color="#FF6B5B", linewidth=2,
         label=f"{case2['span']} slices")
plt.axvline(0, color="#999", linestyle="--")
plt.xlabel("slices from prompt"); plt.ylabel("Dice")
plt.ylim(0, 1.05); plt.legend(); plt.tight_layout()
plt.savefig("decay_both.png", dpi=150, bbox_inches="tight")

In [ ]:
fig = plot_slices(case2["volume"], masks2, case2["truth"], rotate=ROTATE, ncols=8,
                  title=f"{case2['name']} — Dice {dice(case2['truth'], masks2):.3f}")
fig.savefig("slices_long.png", dpi=150, bbox_inches="tight")

---

Project: https://github.com/rekalantar/medsam2-3d-ct